In [ ]:
from IPython.lib.security import random
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
import torch
import difflib
from collections import Counter
import random
import torch.nn.functional as F

## We keep the arguments that do not match

In [2]:
def add_differents_arg(path, doc, final_args):
  #print(f'{folder_output}{threshold}/{doc}{llm1}_{llm2}_original_no_matches.xlsx')
  #print(f'{folder_output}{threshold}/{doc}{llm1}_{llm2}_llm_no_matches.xlsx')
  data_llm1 = pd.read_excel(f'{folder_output}{threshold}/{doc}agree_{llm1}_0.6_agree_{llm2}_0.6_original_no_matches.xlsx', sheet_name=sheetname).arg_text
  #data_llm1 = pd.read_excel(f'{folder_output}{threshold}/{doc}{llm1}_{llm2}_original_no_matches.xlsx', sheet_name=sheetname).arg_text
  #data_llm2 = pd.read_excel(f'{folder_output}{threshold}/{doc}agree_{llm1}_0.6_agree_{llm2}_0.6_llm_no_matches.xlsx', sheet_name=sheetname).arg_text
  final_args.update(set(data_llm1))
  #final_args.update(set(data_llm2))

## We make groups
Since one argument can be paired with others, we keep the pair that has the most similarity

In [ ]:
def get_claim(arg, claims):
  frec = Counter(claims)
  claim = frec.most_common(1)[0][0]
  if frec.most_common(1)[0][1] == 1: #preferimos la de llm1
    claim = arg
  return claim , [i for i in claims if i != claim]

def clean_premises(premises, claim):
  #quitamos premisas que estan contenidas en otras premisas
  # o que esten en la claim
  final = []
  for prep in premises:
    if prep not in claim:
      for j in premises:
        if prep[:-1] not in j:
          final.append(prep)
          break
  return final

def combine_premises(premises, gold_prem, gold_claim):
  premises = clean_premises(list(premises), gold_claim)
  final = []
  rep = []
  for prep in premises:
    in_gols = False
    tmp = premises.copy()
    tmp.remove(prep) #Encontramos las premisas que se parecen mucho
    if prep not in rep:
      matches = difflib.get_close_matches(prep, tmp, cutoff=0.6)
      rep += matches
      if len(matches) > 0:
        for i in matches:
          if i in gold_prem:
            #si una de ellas es parte del arg encontrado por llm1 nos quedamos con esa
            final.append(i)
            in_gols = True
            break
        if not in_gols:
          ordenadas = sorted(matches, key=len, reverse=True)
          change = False
          for i in ordenadas:
            if ('Article' or 'art') in i: #Si alguna tiene referencia a un artículo la conservamos
              final.append(i) #la más larga de preferencia
              change = True
              break
          if not change:
            final.append(matches[random.randint(0, len(matches)-1)]) #agregamos una al azar (luego intentar otra cosa)
      else:
        final.append(prep) #Es diferente
  return set(final)


def combine(arg, matched, final_args):
  matched = list(set(matched))
  claims , all = [], []
  arg = arg.strip().replace('-', '')
  for i in matched:
    i = i.strip()
    claims.append(i.split('\n')[-1])
    all += i.split('\n')
  claim, no_claim = get_claim(arg.split('\n')[-1], claims)
  all = combine_premises(set(all + no_claim) - set(claim), arg.split('\n'), claim)
  arg = '\n'.join(all) + claim
  final_args.add(arg)


In [5]:
def make_groups(path, doc, final_args):
  #print(f'{folder_output}{threshold}/{doc}{llm1}_{llm2}_original_matches.xlsx')
  args = pd.read_excel(f'{folder_output}{threshold}/{doc}agree_{llm1}_0.6_agree_{llm2}_0.6_original_matches.xlsx', sheet_name=sheetname)
  #args = pd.read_excel(f'{folder_output}{threshold}/{doc}{llm1}_{llm2}_original_matches.xlsx', sheet_name=sheetname)
  #args = pd.read_excel(f'{folder_output}{threshold}/{doc}{llm1}_original_matches.xlsx', sheet_name=sheetname)
  # Sort by similarity in descending order
  if 'similarity' in args.columns:
    #args = args[args['similarity'] < 0.99]
    args = args[args['similarity'] > 0.79]
    args_sorted = args.sort_values(by='similarity', ascending=False)
    # Drop duplicates for 'matched_num', keeping the one with the highest similarity
    unique_matches = args_sorted.drop_duplicates(subset='matched_num', keep='first')
    #unique_matches_m = args_sorted.drop_duplicates(subset='matched_num', keep='first')
    #pairs = np.sort(unique_matches_m[['arg_num', 'matched_num']].values, axis=1)

    #unique_matches = unique_matches_m.iloc[
    #    ~pd.DataFrame(pairs).duplicated().to_numpy()
    #]
    #print(len(unique_matches))

    # Group by 'arg_num' and print the list of 'matched_num'
    grouped_by_arg_num = unique_matches.groupby('arg_num')
    final_groups = []
    for name, group in grouped_by_arg_num:
      final_groups.append(name)
      combine(list(group.arg_text)[0], list(group.matched_text), final_args)
    #add_empty(set(final_groups), args, final_args)

In [ ]:
def make_aggrement(path_root,folder_output, files, llm, output):
  for i in files:
    print(i)
    final_args = set()
    add_differents_arg(folder_output, i, final_args)
    make_groups(folder_output, i, final_args)
    print(len(final_args))
    print('-----')
    df1 = pd.DataFrame({'arg_num':range(0,len(final_args)), 'arg_text': list(final_args)})
    with pd.ExcelWriter(path_root+output, engine='openpyxl', mode='a', if_sheet_exists = 'replace') as writer:
      df1.to_excel(writer, sheet_name=llm+i,index=False)


In [ ]:
path_root = '/content/drive/MyDrive/PhD/ArgumentMining/Answers/'
folder_output = path_root + 'output_arg_similarity_annotator_baai/'
threshold = '0.6'
sheetname = 'Sheet1'
llm1 = 'qwen_qwen' #"agree_llama_qwen_0.8"
llm2 =  'saul_saul' #"saul"


In [ ]:
m = ['00','01','02','04','05','06','10','13','16','17','19', '20', '21','27','29']
g = ['33','35', '38', '40', '41', '42']
o = ['03', '07', '08', '09', '11', '12', '14', '15', '18', '22', '23', '24', '25', '26', '28', '30', '31', '32', '34', '37', '39' ]
s = ['03', '09', '15', '23', '24', '25', '26', '30', '31', '34', '39']
l = ['34', '37', '39']
s = ['07', '08', '09', '11', '12', '18', '22', '28', '32', '34','37', '39']

In [20]:
f'arguments_agree_{llm1}_{llm2}_{threshold}.xlsx'

'arguments_agree_qwen_qwen_saul_saul_0.6.xlsx'

In [21]:
make_aggrement(path_root, folder_output, s,'text_',f'agreement_baai/arguments_agree_{llm1}_{llm2}_{threshold}.xlsx')

07
15
-----
08
24
-----
09
15
-----
11
21
-----
12
11
-----
18
16
-----
22
34
-----
28
19
-----
32
5
-----
34
11
-----
37
13
-----
39
6
-----
